# Setting Up All Artifacts details

In [1]:
## Give appropriate permission to the directory "FOLDER_WITH_ARTIFACTS" you are working with
import os
os.environ['SNPE_ROOT']="/mnt/c/Qualcomm/AIStack/QAIRT/2.31.0.250130" #set up your snpe path here.
os.environ['RAW_FILE_FOLDER']="raw"
os.environ['DLC32']="models/yolo_nas_fp32.dlc"
os.environ['DLC8']="models/yolo_nas_w8a8.dlc"
os.environ['TARGET_INPUT_LIST']="input.txt"
os.environ['ONDEVICE_FOLDER']="yolonas"
os.environ['DEVICE_HOST']="localhost"
os.environ['DEVICE_ID']="728b7a92" #change with your device-id. Use command "adb devices" to get devices names.
os.environ['SNPE_TARGET_ARCH']="aarch64-android"
os.environ['SNPE_TARGET_STL']="libc++_shared.so"

In [2]:
## Note- Use python3.8 or above for generating onnx
# !pip install super-gradients==3.1.2
import torch
from super_gradients.training import models
from super_gradients.common.object_names import Models
import cv2
import numpy as np
import os

  Using cached torch-2.4.1-cp38-cp38-manylinux1_x86_64.whl.metadata (26 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached boto3-1.37.38-py3-none-any.whl.metadata (6.7 kB)
  Using cached jsonschema-4.23.0-py3-none-any.whl.metadata (7.9 kB)
  Using cached Deprecated-1.2.18-py2.py3-none-any.whl.metadata (5.7 kB)
  Using cached opencv_python-4.12.0.88-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (19 kB)
  Using cached scipy-1.10.1-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (58 kB)
  Using cached psutil-7.0.0-cp36-abi3-manylinux_2_12_x86_64.manylinux2010_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (22 kB)
  Using cached tensorboard-2.14.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached coverage-5.3.1-cp38-cp38-manylinux2010_x86_64.whl.metadata (7.7 kB)
  Using cached torchvision-0.19.1-cp38-cp38-manylinux1_x86_64.whl.metadata (6.0 kB)
  Using cached Sphinx-4.0.3-py3-none-any.whl.metadata (8.0 kB)


ModuleNotFoundError: No module named 'super_gradients'

## Getting The dataset
Please, fill coco dataset link in below code block. You might need 10-15 images for quantization.

In [ ]:
!wget http://images.cocodataset.org/zips/val2017.zip -q --show-progress
!unzip val2017.zip
!mkdir "raw"

In [3]:
files = os.listdir('val2017') #val2017 is the datatset folder path. Keeping only 15 images.
for file in files[15:]:
    os.remove("val2017/"+file)

In [4]:
%%bash
rm -rf val2017.zip

## Getting the ONNX Model

In [5]:
os.makedirs('models', exist_ok=True)

### YOLO_NAS_S Model has issues while download. To overcome, follow below steps:
In your python enviroment, update super gradients package (3.1.2) with the modified YOLO_NAS_S download link.

***Code Change 1#***:
In /usr/venv/lib/python3.10/site-packages/super_gradients/training/pretrained_models.py line 47

**Existing Code**: "yolo_nas_s_coco": "https://sghub.deci.ai/models/yolo_nas_s_coco.pth",

**Modify**: "yolo_nas_s_coco": "https://sg-hub-nv.s3.amazonaws.com/models/yolo_nas_s_coco.pth",

***Code Change 2#***:
In /usr/venv/lib/python3.10/site-packages/super_gradient/training/utils/checkpoint_utils.py line 316, Function Name:load_pretrained_weightsline

**Existing code**: unique_filename = url.split("https://sghub.deci.ai/models/")[1].replace("/", "_").replace(" ", "_")

**Modify**: unique_filename = url.split("https://sg-hub-nv.s3.amazonaws.com/models/")[1].replace("/", "_").replace(" ", "_")

In [6]:
model = models.get(Models.YOLO_NAS_S, pretrained_weights="coco")
# Prpare model for conversion
# Input size is in format of [Batch x Channels x Width x Height] where 640 is the standard COCO dataset dimensions
model.eval()
model.prep_model_for_conversion(input_size=[1, 3, 320, 320])
# Create dummy_input
dummy_input = torch.randn([1, 3, 320, 320], device="cpu")
# Convert model to onnx
torch.onnx.export(model, dummy_input, "models/yolo_nas_s.onnx", opset_version=11)

#### Getting the FP32 Model

In [ ]:
%%bash
source $SNPE_ROOT/bin/envsetup.sh
snpe-onnx-to-dlc -i models/yolo_nas_s.onnx -o app/src/main/assets/yolo_nas_s.dlc --out_node 885 --out_node 877

## Preprocessing

In [ ]:
def preprocess(original_image):
    resized_image = cv2.resize(original_image, (320, 320))
    resized_image = resized_image/255
    return resized_image
##Please download Coco2014 dataset and give the path here
dataset_path = "val2017/"
!mkdir -p rawYoloNAS
filenames=[]
for path in os.listdir(dataset_path):
    # check if current path is a file
    if os.path.isfile(os.path.join(dataset_path, path)):
        filenames.append(os.path.join(dataset_path, path))
for filename in filenames:
    original_image = cv2.imread(filename)
    img = preprocess(original_image)
    img = img.astype(np.float32)
    img.tofile("raw/"+filename.split("/")[-1].split(".")[0]+".raw")

In [ ]:
%%bash
find raw -name *.raw > input.txt

## Quantize the DLC

In [ ]:
%%bash
source $SNPE_ROOT/bin/envsetup.sh
snpe-dlc-quantize --input_dlc app/src/main/assets/yolo_nas_s.dlc --input_list input.txt --output_dlc app/src/main/assets/Quant_intermediate_yoloNas_s_320.dlc

<b>- Based on the device where you will execute the model set --htp_socs to sm8750 or sm8650 or sm8550</b>

In [ ]:
%%bash
source $SNPE_ROOT/bin/envsetup.sh
snpe-dlc-graph-prepare --input_dlc app/src/main/assets/Quant_intermediate_yoloNas_s_320.dlc --set_output_tensors=885,877 --htp_socs=sm8750 --output_dlc=app/src/main/assets/Quant_yoloNas_s_320.dlc